In [3]:
import requests
import json
import os
from datetime import datetime, timezone
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import lit, current_timestamp
import uuid

# -------------------------------------------------------
# CONFIG
# -------------------------------------------------------

# Fabric Spark properties are accessed via spark.conf, not os.environ
OTX_API_KEY = spark.conf.get("OTX_API_KEY", None)

# Fallback - if still not found, set manually for now
if not OTX_API_KEY:
    OTX_API_KEY = spark.sparkContext.getConf().get("OTX_API_KEY", None)

PIPELINE_RUN_ID = str(uuid.uuid4())
SOURCE_SYSTEM = "otx_api"

print(f"Pipeline Run ID: {PIPELINE_RUN_ID}")
print(f"API Key loaded: {'Yes' if OTX_API_KEY else 'NO - CHECK ENVIRONMENT'}")
print(f"Timestamp: {datetime.now(timezone.utc)}")

StatementMeta(, d02ac449-62e2-4b11-8dae-b505edc96a1c, 5, Finished, Available, Finished, False)

Pipeline Run ID: fb953c31-6e6a-431c-877f-0a31995fb792
API Key loaded: NO - CHECK ENVIRONMENT
Timestamp: 2026-05-02 14:58:18.192369+00:00


In [8]:
import requests
import json
import os
from datetime import datetime, timezone
from pyspark.sql.types import *
from pyspark.sql.functions import lit, current_timestamp
import uuid

# -------------------------------------------------------
# CONFIG
# DEV: API key set via os.environ
# PROD: Replace with Key Vault reference
# -------------------------------------------------------
os.environ["OTX_API_KEY"] = "SET_YOUR_OTX_API_KEY_HERE"

OTX_API_KEY = os.environ.get("OTX_API_KEY")
PIPELINE_RUN_ID = str(uuid.uuid4())
SOURCE_SYSTEM = "otx_api"
OTX_BASE_URL = "https://otx.alienvault.com/api/v1"

print(f"Pipeline Run ID: {PIPELINE_RUN_ID}")
print(f"API Key loaded: {'Yes' if OTX_API_KEY else 'NO'}")
print(f"Timestamp: {datetime.now(timezone.utc)}")

StatementMeta(, cd445a4b-ad9b-4489-9ae8-7c2837406445, 10, Finished, Available, Finished, False)

Pipeline Run ID: 23e36d74-44ad-413a-b858-cb654a0cc10f
API Key loaded: Yes
Timestamp: 2026-05-03 12:38:16.734390+00:00


In [9]:
# -------------------------------------------------------
# CELL 2 - Fetch OTX Pulses from API
# -------------------------------------------------------
def fetch_otx_pulses(api_key, base_url, limit=50):
    headers = {"X-OTX-API-KEY": api_key}
    url = f"{base_url}/pulses/subscribed?limit={limit}"
    
    all_pulses = []
    page = 1
    
    while url:
        print(f"Fetching page {page}...")
        response = requests.get(url, headers=headers, timeout=30)
        
        if response.status_code != 200:
            print(f"Error: {response.status_code} - {response.text}")
            break
            
        data = response.json()
        pulses = data.get("results", [])
        all_pulses.extend(pulses)
        
        # pagination - get next page url
        url = data.get("next", None)
        page += 1
        
        # safety limit for trial
        if page > 5:
            print("Reached page limit - stopping.")
            break
    
    print(f"\nTotal pulses fetched: {len(all_pulses)}")
    return all_pulses

pulses = fetch_otx_pulses(OTX_API_KEY, OTX_BASE_URL, limit=50)
print(f"First pulse name: {pulses[0]['name']}")


StatementMeta(, cd445a4b-ad9b-4489-9ae8-7c2837406445, 11, Finished, Available, Finished, False)

Fetching page 1...
Fetching page 2...
Fetching page 3...
Fetching page 4...
Fetching page 5...
Reached page limit - stopping.

Total pulses fetched: 250
First pulse name: Weaponizing the Protectors: TeamPCPs Multi-Stage Supply Chain Attack on Security Infrastructure


In [10]:
# -------------------------------------------------------
# CELL 3 - Flatten pulses into bronze_otx_pulses DataFrame
# -------------------------------------------------------
from pyspark.sql import Row
import json

def flatten_pulses(pulses, pipeline_run_id, source_system):
    rows = []
    ingested_at = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    
    for pulse in pulses:
        rows.append({
            "pulse_id": str(pulse.get("id", "")),
            "pulse_name": str(pulse.get("name", "")),
            "description": str(pulse.get("description", "")),
            "author_name": str(pulse.get("author_name", "")),
            "created": str(pulse.get("created", "")),
            "modified": str(pulse.get("modified", "")),
            "tlp": str(pulse.get("tlp", "")),
            "targeted_countries": json.dumps(pulse.get("targeted_countries", [])),
            "malware_families": json.dumps(pulse.get("malware_families", [])),
            "tags": json.dumps(pulse.get("tags", [])),
            "indicator_count": int(pulse.get("indicator_count", 0)),
            "ingested_at": ingested_at,
            "source_system": source_system,
            "pipeline_run_id": pipeline_run_id
        })
    
    return rows

pulse_rows = flatten_pulses(pulses, PIPELINE_RUN_ID, SOURCE_SYSTEM)
df_pulses = spark.createDataFrame(pulse_rows)

print(f"Pulse DataFrame shape: {df_pulses.count()} rows, {len(df_pulses.columns)} columns")
df_pulses.printSchema()

StatementMeta(, cd445a4b-ad9b-4489-9ae8-7c2837406445, 12, Finished, Available, Finished, False)

Pulse DataFrame shape: 250 rows, 14 columns
root
 |-- author_name: string (nullable = true)
 |-- created: string (nullable = true)
 |-- description: string (nullable = true)
 |-- indicator_count: long (nullable = true)
 |-- ingested_at: string (nullable = true)
 |-- malware_families: string (nullable = true)
 |-- modified: string (nullable = true)
 |-- pipeline_run_id: string (nullable = true)
 |-- pulse_id: string (nullable = true)
 |-- pulse_name: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- targeted_countries: string (nullable = true)
 |-- tlp: string (nullable = true)



In [12]:
# -------------------------------------------------------
# CELL 4 - Write to Bronze Delta table: bronze_otx_pulses
# -------------------------------------------------------
bronze_table_pulses = "bronze_otx_pulses"

(df_pulses.write
    .format("delta")
    .mode("append")
    .saveAsTable(bronze_table_pulses)
)

# verify
count = spark.sql(f"SELECT COUNT(*) as total FROM {bronze_table_pulses}").collect()[0]["total"]
print(f"bronze_otx_pulses: {count} rows written successfully")

StatementMeta(, cd445a4b-ad9b-4489-9ae8-7c2837406445, 14, Finished, Available, Finished, False)

bronze_otx_pulses: 250 rows written successfully


In [13]:
# -------------------------------------------------------
# CELL 5 - Flatten indicators into bronze_otx_indicators
# -------------------------------------------------------
def flatten_indicators(pulses, pipeline_run_id, source_system):
    rows = []
    ingested_at = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    
    for pulse in pulses:
        pulse_id = str(pulse.get("id", ""))
        indicators = pulse.get("indicators", [])
        
        for ind in indicators:
            rows.append({
                "indicator_id": str(ind.get("id", "")),
                "pulse_id": pulse_id,
                "indicator": str(ind.get("indicator", "")),
                "type": str(ind.get("type", "")),
                "created": str(ind.get("created", "")),
                "is_active": int(ind.get("is_active", 0)),
                "expiration": str(ind.get("expiration", "")),
                "title": str(ind.get("title", "")),
                "description": str(ind.get("description", "")),
                "ingested_at": ingested_at,
                "source_system": source_system,
                "pipeline_run_id": pipeline_run_id
            })
    
    return rows

indicator_rows = flatten_indicators(pulses, PIPELINE_RUN_ID, SOURCE_SYSTEM)
df_indicators = spark.createDataFrame(indicator_rows)

print(f"Indicators DataFrame: {df_indicators.count()} rows, {len(df_indicators.columns)} columns")
df_indicators.printSchema()

StatementMeta(, cd445a4b-ad9b-4489-9ae8-7c2837406445, 15, Finished, Available, Finished, False)

Indicators DataFrame: 6881 rows, 12 columns
root
 |-- created: string (nullable = true)
 |-- description: string (nullable = true)
 |-- expiration: string (nullable = true)
 |-- indicator: string (nullable = true)
 |-- indicator_id: string (nullable = true)
 |-- ingested_at: string (nullable = true)
 |-- is_active: long (nullable = true)
 |-- pipeline_run_id: string (nullable = true)
 |-- pulse_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- title: string (nullable = true)
 |-- type: string (nullable = true)



In [14]:
# -------------------------------------------------------
# CELL 6 - Write to Bronze Delta table: bronze_otx_indicators
# -------------------------------------------------------
bronze_table_indicators = "bronze_otx_indicators"

(df_indicators.write
    .format("delta")
    .mode("append")
    .saveAsTable(bronze_table_indicators)
)

count = spark.sql(f"SELECT COUNT(*) as total FROM {bronze_table_indicators}").collect()[0]["total"]
print(f"bronze_otx_indicators: {count} rows written successfully")

StatementMeta(, cd445a4b-ad9b-4489-9ae8-7c2837406445, 16, Finished, Available, Finished, False)

bronze_otx_indicators: 6881 rows written successfully


In [15]:
# -------------------------------------------------------
# CELL 7 - Pipeline logging
# -------------------------------------------------------
from pyspark.sql import Row

log_row = [{
    "pipeline_run_id": PIPELINE_RUN_ID,
    "pipeline_name": "nb_bronze_otx_ingestion",
    "source_system": SOURCE_SYSTEM,
    "table_name": "bronze_otx_pulses, bronze_otx_indicators",
    "rows_loaded_pulses": df_pulses.count(),
    "rows_loaded_indicators": df_indicators.count(),
    "status": "SUCCESS",
    "run_timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
    "notes": "Initial Bronze load - 5 pages OTX subscribed pulses"
}]

df_log = spark.createDataFrame(log_row)

(df_log.write
    .format("delta")
    .mode("append")
    .saveAsTable("pipeline_log")
)

print(f"Pipeline log written for run: {PIPELINE_RUN_ID}")
spark.sql("SELECT * FROM pipeline_log").show(truncate=False)

StatementMeta(, cd445a4b-ad9b-4489-9ae8-7c2837406445, 17, Finished, Available, Finished, False)

Pipeline log written for run: 23e36d74-44ad-413a-b858-cb654a0cc10f
+---------------------------------------------------+-----------------------+------------------------------------+----------------------+------------------+-------------------+-------------+-------+----------------------------------------+
|notes                                              |pipeline_name          |pipeline_run_id                     |rows_loaded_indicators|rows_loaded_pulses|run_timestamp      |source_system|status |table_name                              |
+---------------------------------------------------+-----------------------+------------------------------------+----------------------+------------------+-------------------+-------------+-------+----------------------------------------+
|Initial Bronze load - 5 pages OTX subscribed pulses|nb_bronze_otx_ingestion|23e36d74-44ad-413a-b858-cb654a0cc10f|6881                  |250               |2026-05-03 12:46:42|otx_api      |SUCCESS|bronze_otx_puls

In [16]:
# -------------------------------------------------------
# CELL 8 - Bronze layer verification
# -------------------------------------------------------
tables = ["bronze_otx_pulses", "bronze_otx_indicators", "pipeline_log"]

print("=== Bronze Layer Verification ===\n")
for table in tables:
    count = spark.sql(f"SELECT COUNT(*) as total FROM {table}").collect()[0]["total"]
    print(f"✓ {table}: {count} rows")

print("\n=== Sample from bronze_otx_pulses ===")
spark.sql("""
    SELECT pulse_id, pulse_name, indicator_count, tlp, ingested_at 
    FROM bronze_otx_pulses 
    LIMIT 3
""").show(truncate=50)

print("\n=== Sample from bronze_otx_indicators ===")
spark.sql("""
    SELECT indicator_id, pulse_id, indicator, type, is_active 
    FROM bronze_otx_indicators 
    LIMIT 3
""").show(truncate=50)


StatementMeta(, cd445a4b-ad9b-4489-9ae8-7c2837406445, 18, Finished, Available, Finished, False)

=== Bronze Layer Verification ===

✓ bronze_otx_pulses: 250 rows
✓ bronze_otx_indicators: 6881 rows
✓ pipeline_log: 1 rows

=== Sample from bronze_otx_pulses ===
+------------------------+--------------------------------------------------+---------------+-----+-------------------+
|                pulse_id|                                        pulse_name|indicator_count|  tlp|        ingested_at|
+------------------------+--------------------------------------------------+---------------+-----+-------------------+
|69bd18a61f631ff045510990|CVE-2026-33017: How attackers compromised Langf...|              0|white|2026-05-03 12:39:28|
|69bd01b20154ae405e9187fe|Copyright Lures Mask a Multi-Stage PureLog Stea...|              0|white|2026-05-03 12:39:28|
|69bd045137b178c16714dcf6|               An Overview of The Gentlemen's TTPs|              0|white|2026-05-03 12:39:28|
+------------------------+--------------------------------------------------+---------------+-----+-------------------